# UEMR chốt, file 03: benchmark zero-shot YouCook2, cả pipeline rồi chấm một lần

Theo `docs/uemr_chot/UEMR_CHOT_03_BENCHMARK.md`. **Không train, không chọn, không chỉnh gì ở đây:** θ, `L_min`, `L_fill`, β\*, phương pháp
chính đọc từ `configs/uemr_final.json` (khoá ở file 02, ô 6 kiểm sha256). Một lần Run all đi hết pipeline:

| Ô | Việc | Cùng cách với lúc train / dev |
|---|---|---|
| 3, 4 | kiểm kê, kiểm trùng ID, câu hỏi (câu của bước GT, nguyên văn) | |
| 5 | mp4 → đặc trưng InternVideo2 K400 f16 + BEATs | script HanLi đã tạo đặc trưng train UniAV; CHECK tái lập cos ≥ 0,99 trên 5 video HT-Step |
| 6 | đặc trưng → UniAV → đoạn R6 → R8 (đoạn lấp), R2, R3, vector `h_i` | HanLi `77fc4fa`, checkpoint của config, article đoán, `HTStepDataset` không sửa |
| 7 | khoá `benchmark_protocol.json` (in sha256) **trước** khi chấm | |
| 8 | Omni `best`: mọi dòng + text; Omni pt: R0, R8 + text | `omni_retrieval.encode`, mp4 + `timestamps`, như cột (b) của dev |
| 9 | **chấm một lần**: mọi dòng × cột, cặp + CI, tiêu chí K1–K6 | `uemr_htstep.final_bench` (cùng hàm chấm với dev) |
| 10 | tóm tắt cho Nhật ký | |

- Colab Secrets: `HF_TOKEN` (tài khoản đã chấp nhận licence của `OpenGVLab/InternVideo2-Stage1-1B-224p-K400` và
  `OpenGVLab/InternVideo2-Stage2-6B-Audio`). GPU A100, RAM cao (ô 9 cần ~25 GB RAM).
- **Kho = mọi video YouCook2 có annotation và có mp4 (train + val, ~1.500 video)**, quyết định của user 2026-11-13: không model nào của
  plan chốt train trên YouCook2. Annotation lấy từ repo (`configs/yc2/`), đối chiếu với bản chính thức nếu tải được.
- Video YouCook2 chỉ **đọc** từ Drive `Colab Notebooks/code KL/Omni/data/YouCookII/videos`. **Không ghi gì lên Drive:** đầu ra nằm ở
  `/content/uemr_chot_run/bench/yc2`, tự đẩy lên HF `GraceFeel/uemr-bench-yc2` (private) sau mỗi ô và mỗi 20 phút; mất runtime thì Run all
  lại, ô 1 tải lại từ HF rồi chạy tiếp (đặc trưng, đoạn, `h_i`, vector Omni đã có đều được bỏ qua).
- **Chạy thử trước:** `SMOKE = True` (5 video, không đẩy HF, số chấm trên 5 video vô nghĩa, chỉ để thử code). Xong đổi `SMOKE = False`, Run all.
- Ô 9 chỉ chấm khi chưa có `results/score/criteria.json`; đã có thì in lại, **không chấm lại**.

In [ ]:
# ===== 0. Cấu hình =====
FUSHION_REF = 'c7e020e1f89b901bb10430b4e2bfcacbc5aa3fb1'   # commit trên nhánh htstep (user push trước khi chạy)
BENCHMARKS  = ['yc2']    # 'anet' chỉ khi user thêm (plan 03); notebook này mới có 'yc2'
SMOKE       = True       # True: 5 video, không đẩy HF. Xong đổi False rồi Run all
YC2_SUBSET  = None       # None: mọi video có annotation (train + val); 'validation': chỉ split validation như plan gốc
HF_OUT, HF_DIR = 'GraceFeel/uemr-bench-yc2', 'yc2'      # dataset private: mọi đầu ra của notebook (resume đọc lại từ đây)
RUN_FEATS   = True       # ô 5: trích đặc trưng K400 + BEATs (GPU)
RUN_OMNI    = True       # ô 8: encode Omni (A100)
OMNI_TAGS   = ['best', 'pt']
SEED        = 0
SEC_REF     = 0.32       # s/clip Omni trên A100 (lượt dev 1: 0.314)
HANLI_REF, OMNI_FIX_REF = '77fc4fa', 'ab772eb'
PROTO1_SHA16 = 'a4ca43db9829842a'   # protocol lượt dev 1: nguồn hash của adapter, danh sách train, kho dev
CONFIG_SHA  = 'c9840ffbc964a4b0ed70baee157213adaaa6526178316c80a8435e7d2fcdbde3'   # configs/uemr_final.json đã khoá (file 02 §7)
assert BENCHMARKS == ['yc2'], 'notebook này mới chuẩn bị yc2'
FLAGS = {k: v for k, v in dict(globals()).items() if k.startswith('RUN_') or k in ('FUSHION_REF', 'BENCHMARKS', 'SMOKE', 'OMNI_TAGS', 'SEED', 'YC2_SUBSET', 'HF_OUT')}

In [ ]:
# ===== 1. Drive (chỉ đọc), HF (đầu ra), code (Fushion, HanLi, Omni-fix), thư viện, unit test =====
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, glob, shutil, subprocess, time, hashlib, platform, csv, tarfile, urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
os.environ.pop('HF_XET_HIGH_PERFORMANCE', None)
T_START = time.time()
FUSHION, HANLI, OMNI_FIX = '/content/code/Fushion_UEMR', '/content/code/HanLi-Retriever', '/content/code/Omni-fix'
os.makedirs('/content/code', exist_ok=True)

def clone(url, path, ref, *branch):
    os.chdir('/content')
    shutil.rmtree(path, ignore_errors=True)
    subprocess.run(['git', 'clone', '-q', *branch, url, path], check=True)
    r = subprocess.run(['git', '-C', path, 'checkout', '-q', ref], capture_output=True, text=True)
    assert r.returncode == 0, f'không checkout được {ref} của {url}: {r.stderr} (đã push chưa?)'
    return subprocess.run(['git', '-C', path, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip()

COMMIT = clone('https://github.com/chien24/Fushion_UEMR.git', FUSHION, FUSHION_REF)
HANLI_COMMIT = clone('https://github.com/im-xiaoming/HanLi-Retriever.git', HANLI, HANLI_REF)
OMNI_COMMIT = clone('https://github.com/im-xiaoming/Omni-Retriever-Custom.git', OMNI_FIX, OMNI_FIX_REF, '-b', 'htstep_lora')
print('Fushion :', COMMIT); print('HanLi   :', HANLI_COMMIT); print('Omni-fix:', OMNI_COMMIT)
if FUSHION not in sys.path:
    sys.path.insert(0, FUSHION)
os.environ['PYTHONPATH'] = FUSHION + ':' + os.environ.get('PYTHONPATH', '')
os.chdir(FUSHION)

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr)[-3000:]
    if out.strip():
        print(out)
    if check and r.returncode:
        raise RuntimeError(f'lệnh lỗi ({r.returncode}): {cmd[:200]}')
    return r

def check(name, ok, detail=''):
    print(f"CHECK {name}: {'OK' if ok else 'FAIL'}" + (f' | {detail}' if detail else ''))
    assert ok, f'CHECK {name} FAIL: {detail}'

sh(f'pip install -q -r {FUSHION}/requirements.txt')
sh('pip uninstall -y -q torchao', check=False)
import numpy as np, pandas as pd, torch
from IPython.display import display
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có'
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0.0
ENV = {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__, 'gpu': gpu, 'gpu_GB': round(GPU_GB, 1),
       'fushion': COMMIT, 'hanli': HANLI_COMMIT, 'omni_fix': OMNI_COMMIT, 'smoke': SMOKE}
print(json.dumps(ENV, indent=1, ensure_ascii=False))
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_bench.py', 'tests/test_uemr_htstep.py', '-q', '-p', 'no:cacheprovider'], cwd=FUSHION,
                   capture_output=True, text=True, env={**os.environ, 'UNIAV_REPO': HANLI})
print(r.stdout[-1200:])
check('unit test', r.returncode == 0 and ' skipped' not in r.stdout.splitlines()[-1], r.stdout.splitlines()[-1] if r.stdout else r.stderr[-300:])

def sha256(path, bs=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(bs), b''):
            h.update(chunk)
    return h.hexdigest()

def wj(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=str)
    os.replace(tmp, path)

def rj(path, default=None):
    return json.load(open(path, encoding='utf-8')) if os.path.exists(path) else default

DRIVE = '/content/drive/MyDrive'
YC    = f'{DRIVE}/Colab Notebooks/code KL/Omni/data/YouCookII'                # videos/ (chỉ đọc)
B     = '/content/uemr_chot_run/bench/' + HF_DIR + ('_smoke' if SMOKE else '')    # mọi đầu ra của bộ yc2: /content, không ghi Drive
RES   = f'{B}/results'
RUN_DRIVE = f'{DRIVE}/omniretriever/htstep_lora/lora_bs32'
A, VIDEOS = '/content/bench', '/content/videos'      # /content/videos: bản mp4 decord đọc được (VIDEO_ROOT của encode)
import threading
from huggingface_hub import HfApi, snapshot_download
API, _SYNC_LOCK = HfApi(), threading.Lock()
if SMOKE:
    shutil.rmtree(B, ignore_errors=True)
else:      # resume: lấy lại mọi đầu ra đã đẩy lên HF
    API.create_repo(HF_OUT, repo_type='dataset', private=True, exist_ok=True)
    snapshot_download(HF_OUT, repo_type='dataset', allow_patterns=[f'{HF_DIR}/*'], local_dir=os.path.dirname(B))
    for f in glob.glob(f'{B}/**/*.npz', recursive=True):      # file đẩy lên giữa lúc đang ghi: xoá để làm lại
        try:
            np.load(f).close()
        except Exception:      # noqa: BLE001
            print('npz hỏng, xoá:', f); os.remove(f)
    print('đã có trên HF:', sum(len(fs) for _, _, fs in os.walk(B)), 'file')
def sync_up(msg):
    if SMOKE:
        return
    with _SYNC_LOCK:
        API.upload_folder(repo_id=HF_OUT, repo_type='dataset', folder_path=B, path_in_repo=HF_DIR, commit_message=f'{msg} (Fushion {COMMIT[:7]})',
                          ignore_patterns=['*.tmp', '*.tmp.npz', '*.part', '.cache/*'])
    print('đã đẩy HF:', msg)
def _sync_loop():      # mất runtime giữa ô dài thì mất nhiều nhất 20 phút
    while True:
        time.sleep(1200)
        try:
            sync_up('tự động 20 phút')
        except Exception as e:      # noqa: BLE001 - lỗi mạng không được làm dừng ô đang chạy
            print('sync HF lỗi (thử lại sau 20 phút):', e)
if not SMOKE and not globals().get('_SYNC_THREAD'):
    _SYNC_THREAD = threading.Thread(target=_sync_loop, daemon=True); _SYNC_THREAD.start()
for d in (B, RES, A, VIDEOS):
    os.makedirs(d, exist_ok=True)
INV = rj(f'{RES}/inventory.json', {})
def note(**kw):
    INV.update(kw); wj(f'{RES}/inventory.json', INV)

In [ ]:
# ===== 2. Adapter best và pt; hash theo protocol lượt dev 1 =====
PR = json.load(open(f'{FUSHION}/configs/htstep_uemr_protocol.json', encoding='utf-8'))
check('protocol lượt dev 1 (nguồn hash)', sha256(f'{FUSHION}/configs/htstep_uemr_protocol.json')[:16] == PROTO1_SHA16)
D, M = PR['data'], PR['models']
BEST_ADAPTER, PT_ADAPTER = '/content/adapters/best', '/content/adapters/omniretriever-7b'
if not os.path.exists(f'{BEST_ADAPTER}/adapter_model.safetensors'):
    src = f'{RUN_DRIVE}/eval/adapter_best'
    if os.path.exists(f'{src}/adapter_model.safetensors'):
        shutil.copytree(src, BEST_ADAPTER, dirs_exist_ok=True)
    else:
        sh("hf download GraceFeel/omni_lora_htstep --include 'lora_bs32/adapter_best/*' --local-dir /content/adapters/_hf")
        shutil.copytree('/content/adapters/_hf/lora_bs32/adapter_best', BEST_ADAPTER, dirs_exist_ok=True)
if 'pt' in OMNI_TAGS and not os.path.exists(f'{PT_ADAPTER}/adapter_model.safetensors'):
    sh(f"hf download {M['omni_pt']['repo']} --include 'omniretriever-7b/*' --local-dir /content/adapters")
ADAPTERS = {'best': BEST_ADAPTER, 'pt': PT_ADAPTER}
ADAPTER_SHA = {t: sha256(f'{ADAPTERS[t]}/adapter_model.safetensors') for t in OMNI_TAGS}
check('sha256 adapter best = 5e9899aa…', ADAPTER_SHA['best'] == M['omni_best']['sha256'], ADAPTER_SHA['best'][:16])
if 'pt' in OMNI_TAGS:
    check('sha256 adapter pt = 31e81f1c…', ADAPTER_SHA['pt'] == M['omni_pt']['sha256'], ADAPTER_SHA['pt'][:16])

In [ ]:
# ===== 3. Plan §1: kiểm kê từng bước lọc, kiểm trùng YouTube ID với train của hai model và kho dev =====
import importlib.util as _ilu
from uemr_htstep import bench, data as hd
_spec = _ilu.spec_from_file_location('htstep_io', f'{OMNI_FIX}/src/omniretriever/data/htstep_io.py')
hio = _ilu.module_from_spec(_spec); _spec.loader.exec_module(hio)
t0 = time.time()
# annotation: bản chính thức của YouCook2 nếu tải được; bản trong repo (đã dùng ở các lượt YC2 cũ, split train tách thêm 'dev') phải trùng nó
DRIVE_ANN = f'{FUSHION}/configs/yc2/youcookii_annotations_uemr.json'
drive_val = bench.yc2(json.load(open(DRIVE_ANN, encoding='utf-8'))['database'], YC2_SUBSET)
OFFICIAL = f'{A}/youcookii_annotations_trainval.json'
try:
    if not os.path.exists(OFFICIAL):
        urllib.request.urlretrieve('http://youcook2.eecs.umich.edu/static/YouCookII/youcookii_annotations_trainval.tar.gz', f'{A}/ann.tar.gz')
        with tarfile.open(f'{A}/ann.tar.gz') as tf:
            m = next(x for x in tf if x.name.endswith('youcookii_annotations_trainval.json'))
            with tf.extractfile(m) as src, open(OFFICIAL, 'wb') as dst:
                shutil.copyfileobj(src, dst)
    BENCH = bench.yc2(json.load(open(OFFICIAL, encoding='utf-8'))['database'], YC2_SUBSET)
    ANN_SRC = {'file': 'youcookii_annotations_trainval.json (youcook2.eecs.umich.edu)', 'sha256': sha256(OFFICIAL), 'official': True}
    core = lambda b: {v: [(float(a['segment'][0]), float(a['segment'][1]), a['sentence']) for a in x['annotations']] for v, x in b.items()}
    check('annotation trong repo = bản chính thức (video, đoạn, câu)', core(BENCH) == core(drive_val), f'{len(BENCH)} vs {len(drive_val)} video')
except (OSError, StopIteration, tarfile.TarError) as e:      # trang YouCook2 không vào được: dùng bản repo, ghi rõ chưa đối chiếu
    print(f'KHÔNG tải được annotation chính thức ({type(e).__name__}: {e}); dùng bản repo, CHƯA đối chiếu với bản chính thức')
    BENCH, ANN_SRC = drive_val, {'file': DRIVE_ANN, 'sha256': sha256(DRIVE_ANN), 'official': False}
N_ANN = {None: 1790, 'validation': 457}[YC2_SUBSET]      # trainval chính thức: 1.333 train + 457 val
check(f'YouCook2 ({YC2_SUBSET or "train + val"}) có {N_ANN} video', len(BENCH) == N_ANN, str(len(BENCH)))

IDS = json.load(open(f"{FUSHION}/{D['train_ids']['file']}", encoding='utf-8'))
check('sha256 configs/htstep_train_ids.json', sha256(f"{FUSHION}/{D['train_ids']['file']}") == D['train_ids']['sha256'])
OVER = bench.overlap(BENCH, uniav_train=IDS['uniav_train'], omni_train=IDS['omni_train'], dev_510=D['gallery'])
print('trùng YouTube ID:', {k: len(v) for k, v in OVER.items()}, {k: v[:5] for k, v in OVER.items() if v})

on_drive = {f[:-4] for f in os.listdir(f'{YC}/videos') if f.endswith('.mp4')} & set(BENCH)
if SMOKE:
    on_drive = set(sorted(on_drive)[:5])
def localize(v):
    """mp4 từ Drive về /content/yc2_raw (resume); bản decord đọc được ở /content/videos (AV1 chuyển H.264 không mất mát như lượt dev 1)."""
    raw, dst, info = f'/content/yc2_raw/{v}.mp4', f'{VIDEOS}/{v}.mp4', {}
    try:
        if not (os.path.exists(raw) and os.path.getsize(raw) == os.path.getsize(f'{YC}/videos/{v}.mp4')):
            shutil.copy2(f'{YC}/videos/{v}.mp4', raw + '.part'); os.replace(raw + '.part', raw)
        p = hio.probe(raw)
        info['codec'] = p['codec']
        info['audio'] = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'a:0', '-show_entries', 'stream=codec_name', '-of', 'csv=p=0', raw],
                                       capture_output=True, text=True).stdout.strip()
        if not (os.path.exists(dst) and os.path.getsize(dst)):
            if os.path.lexists(dst):
                os.remove(dst)
            if p['codec'] == 'av1':
                ok, why, _ = hio.transcode(raw, dst + '.h264.mp4', p)
                assert ok, f'transcode: {why}'
                os.replace(dst + '.h264.mp4', dst)
            else:
                os.symlink(raw, dst)
        from decord import VideoReader
        vr = VideoReader(dst, num_threads=1)
        n, fps = len(vr), float(vr.get_avg_fps())
        vr[0]; vr[n - 1]
        info.update(frames=n, fps=fps, duration=n / fps)      # thời lượng = khung / fps trung bình, như loader
    except Exception as e:     # noqa: BLE001 - một video lỗi không làm mất các video khác
        info['error'] = f'{type(e).__name__}: {e}'
    return v, info
os.makedirs('/content/yc2_raw', exist_ok=True)
free = shutil.disk_usage('/content').free / 2**30
size = sum(os.path.getsize(f'{YC}/videos/{v}.mp4') for v in on_drive) / 2**30
check('đĩa /content đủ cho video', free >= size + 30, f'video {size:.0f} GB, trống {free:.0f} GB')
with ThreadPoolExecutor(8) as ex:
    PROBE = dict(ex.map(localize, sorted(on_drive)))
readable = {v for v, p in PROBE.items() if 'error' not in p}
with_audio = {v for v, p in PROBE.items() if p.get('audio')}
VIDS, COUNTS = bench.funnel(BENCH, on_drive, with_audio, readable, {v for ids in OVER.values() for v in ids})
DUR = {v: round(PROBE[v]['duration'], 3) for v in VIDS}
print('từng bước lọc:', COUNTS, '| codec:', dict(Counter(p.get('codec', 'lỗi') for p in PROBE.values())))
bad = {v: p['error'] for v, p in PROBE.items() if 'error' in p}
print('không đọc được:', bad)
gap = {v: abs(DUR[v] - float(BENCH[v]['duration'])) for v in VIDS}
print('lệch thời lượng mp4 so với annotation: trung vị %.2f s, lớn nhất %.2f s (%s)' % (np.median(list(gap.values())), max(gap.values()), max(gap, key=gap.get)))
LIMITS = ['Dữ liệu pretrain của Omni gốc (InternVid, InternVid-FLT, Panda-70M, PVD) cũng lấy từ YouTube và không kiểm trùng hết được.',
          f"{COUNTS['annotation'] - COUNTS['tải được']} video có annotation không có mp4 (không tải được từ YouTube từ các lượt trước)." if not SMOKE else 'SMOKE: 5 video.']
VIDS_SHA = hashlib.sha256(('\n'.join(VIDS) + '\n').encode()).hexdigest()
note(annotation={**ANN_SRC, 'subset': YC2_SUBSET or 'training + validation'}, filter_counts=COUNTS, overlap={k: v for k, v in OVER.items()}, overlap_checked_against={'uniav_train': len(IDS['uniav_train']), 'omni_train': len(IDS['omni_train']), 'dev_510': len(D['gallery'])},
     videos=VIDS, videos_sha256=VIDS_SHA, durations=DUR, unreadable=bad, codecs=dict(Counter(p.get('codec', 'lỗi') for p in PROBE.values())), limits=LIMITS)
check('còn video sau khi lọc', len(VIDS) > 0, str(COUNTS))
print(f'kho yc2: {len(VIDS)} video, sha256 danh sách {VIDS_SHA[:16]} | {time.time() - t0:.0f} s')
sync_up('ô 3: kiểm kê')

In [ ]:
# ===== 4. Plan §3: câu hỏi = câu mô tả của từng bước GT (nguyên văn), multi-positive, strict; các dòng không cần UniAV (R0, R7) =====
CAPS, DROPPED = bench.queries(BENCH, VIDS, DUR)
GT, R0 = hd.gt_partition(CAPS), bench.r0(DUR)
HSTAT = hd.headline_stats(CAPS)
_vids = {}
for c in CAPS:
    _vids.setdefault(c['headline'], set()).add(c['video_id'])
shared = sum(len(_vids[c['headline']]) >= 2 for c in CAPS)
print(f'câu hỏi: {len(CAPS)} | bước không còn cửa sổ trong mp4 (bỏ): {DROPPED} | text khác nhau: {HSTAT["distinct texts"]} | câu có text trùng ở >= 2 video: {shared}')
display(pd.DataFrame(HSTAT['top'][:10]))
check('mọi video của kho có câu hỏi', {c['video_id'] for c in CAPS} == set(VIDS), str(sorted(set(VIDS) - {c['video_id'] for c in CAPS})[:5]))
wj(f'{RES}/queries.json', CAPS)
note(queries=len(CAPS), queries_dropped=DROPPED, queries_sha256=sha256(f'{RES}/queries.json'), shared_text={k: v for k, v in HSTAT.items() if k != 'top'}, queries_shared_text=shared,
     rules={'query': 'câu của bước GT, nguyên văn; không ghép tên công thức / activity', 'normalize': 'uemr_htstep.data.normalize (lower, bỏ dấu câu, gộp khoảng trắng); phải trùng luật trong configs/uemr_final.json khi khoá',
            'multi_positive': 'mọi video của kho có bước cùng text đã chuẩn hoá', 'strict': 'video gốc của câu', 'vcmr': 'tIoU >= 0.5 với một bước có text đó trong video đứng đầu',
            'window': 'đoạn GT cắt theo thời lượng mp4, làm tròn 0.01 s'})
sync_up('ô 4: câu hỏi')

In [ ]:
# ===== 5. Plan §2: đặc trưng UniAV (InternVideo2 K400 f16 + BEATs), CHECK tái lập trên 5 video kho HT-Step, rồi trích cho benchmark =====
# Script: HanLi tools/pipeline/extract_features.py (--model k400_f16 mặc định), model card nguyenminh04/uniav-htstep-omni ghi đúng script này.
from huggingface_hub import hf_hub_download
FEATS = f'{B}/feats_k400'      # <video_id>.npz: k768 + a768, một dòng mỗi giây (htstep.py nội suy lên lưới 256 bước khi đọc)
os.makedirs(FEATS, exist_ok=True)
if RUN_FEATS:
    t0 = time.time()
    IV = '/content/InternVideo'
    if not os.path.isdir(f'{IV}/.git'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/OpenGVLab/InternVideo', IV], check=True)
    IV_COMMIT = subprocess.run(['git', '-C', IV, 'log', '-1', '--format=%h %ad', '--date=short'], capture_output=True, text=True).stdout.strip()
    sh('pip install -q timm einops')
    CK = '/content/ckpt/internvideo2'
    for repo, f in (('OpenGVLab/InternVideo2-Stage1-1B-224p-K400', '1B_ft_k710_ft_k400_f16.pth'), ('OpenGVLab/InternVideo2-Stage2-6B-Audio', 'audio_6b.pth')):
        hf_hub_download(repo, f, local_dir=CK)
    CK_SHA = {f: sha256(f'{CK}/{f}') for f in ('1B_ft_k710_ft_k400_f16.pth', 'audio_6b.pth')}
    EXTRACT_COMMIT = subprocess.run(['git', '-C', HANLI, 'log', '-1', '--format=%h %ad %s', '--date=short', '--', 'tools/pipeline/extract_features.py'], capture_output=True, text=True).stdout.strip()
    print('extract_features.py:', EXTRACT_COMMIT, '| InternVideo:', IV_COMMIT)
    def extract(videos, out, log, extra=''):
        get_ipython().system(f'cd {HANLI} && python tools/pipeline/extract_features.py --csv "" --videos "{videos}" --out "{out}" --repo {IV} --ckpt-dir {CK} '
                             f'--audio-ckpt {CK}/audio_6b.pth --batch-size 32 --decode-ahead 8 {extra} 2>&1 | tee -a "{log}"')      # batch 32, decode 8: như colab_dl_extract.sh

    # CHECK tái lập: 5 video kho HT-Step ngắn nhất có trong tar đặc trưng mẫu, trích lại từ mp4 gốc, so với đặc trưng đã dùng để train / chạy UniAV
    REP = f'{A}/repro'
    os.makedirs(f'{REP}/ref', exist_ok=True); os.makedirs(f'{REP}/videos', exist_ok=True)
    tar = hf_hub_download('nguyenminh04/htstep-feats-k400', 'feats/htstep_feats_1000.tar', repo_type='dataset', local_dir=f'{A}/feat_tars')      # 90 video kho
    with tarfile.open(tar) as tf:
        for m in tf:
            if m.isfile() and m.name.endswith('.npz') and m.size > 0:
                with tf.extractfile(m) as src, open(f'{REP}/ref/{os.path.basename(m.name)}', 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    items = hd.read_jsonl(hf_hub_download(D['items_val']['hf'], D['items_val']['file'], repo_type='dataset', local_dir=A))
    SRC = {r['video_id']: r['source'] for r in items}
    TARI = {r['video_id']: (int(r['offset_data']), int(r['size'])) for r in csv.DictReader(open(f'{OMNI_FIX}/data/htstep/tar_index.csv', newline='', encoding='utf-8'))}
    have_ref = {f[:-4] for f in os.listdir(f'{REP}/ref')} & set(D['gallery']) & set(SRC)
    REP_VIDS = sorted(have_ref, key=lambda v: (D['durations'][v], v))[:5]
    for v in REP_VIDS:
        dst, s = f'{REP}/videos/{v}.mp4', SRC[v]
        if not (os.path.exists(dst) and os.path.getsize(dst)):
            hio.range_get(s['tar'], *TARI[v], dst, repo=s['repo']) if 'tar' in s else hio.hf_get(s['repo'], s['file'], dst)
    shutil.rmtree(f'{REP}/out', ignore_errors=True)
    extract(f'{REP}/videos', f'{REP}/out', f'{B}/extract.log')
    def row_cos(a, b):
        t = min(len(a), len(b))
        a, b = a[:t].astype(np.float32), b[:t].astype(np.float32)
        na, nb = np.linalg.norm(a, axis=1), np.linalg.norm(b, axis=1)
        both0 = (na < 1e-6) & (nb < 1e-6)                      # video không có tiếng: a768 toàn 0 ở cả hai
        c = (a * b).sum(1) / np.maximum(na * nb, 1e-12)
        return float(np.where(both0, 1.0, c).mean())
    REPRO = {}
    for v in REP_VIDS:
        check(f'trích lại được {v}', os.path.exists(f'{REP}/out/{v}.npz'))
        with np.load(f'{REP}/out/{v}.npz') as x, np.load(f'{REP}/ref/{v}.npz') as y:
            REPRO[v] = {'k768': row_cos(x['k768'], y['k768']), 'a768': row_cos(x['a768'], y['a768']), 'rows': [len(x['k768']), len(y['k768'])]}
    display(pd.DataFrame(REPRO).T)
    note(features={'script': 'HanLi-Retriever tools/pipeline/extract_features.py --model k400_f16', 'script_commit': EXTRACT_COMMIT, 'hanli': HANLI_COMMIT, 'internvideo': IV_COMMIT,
                   'ckpt_sha256': CK_SHA, 'repro': REPRO})
    check('tái lập đặc trưng: cos từng dòng trung bình >= 0.99 cho k768 và a768 (5 video)', all(r['k768'] >= 0.99 and r['a768'] >= 0.99 for r in REPRO.values()),
          str({v: r for v, r in REPRO.items() if min(r['k768'], r['a768']) < 0.99}))

    # benchmark: đo trên 5 video rồi chạy hết (resume: video đã có npz trong FEATS thì bỏ qua)
    SRCV = f'{A}/feat_src'
    shutil.rmtree(SRCV, ignore_errors=True); os.makedirs(SRCV)
    for v in VIDS:
        os.symlink(f'/content/yc2_raw/{v}.mp4', f'{SRCV}/{v}.mp4')
    left = [v for v in VIDS if not os.path.exists(f'{FEATS}/{v}.npz')]
    total_sec = sum(DUR[v] for v in left)
    print(f'video cần trích: {len(left)} / {len(VIDS)}, {total_sec / 3600:.1f} giờ video')
    if left:
        t1 = time.time()
        extract(SRCV, FEATS, f'{B}/extract.log', '--limit 5')
        got = [v for v in left if os.path.exists(f'{FEATS}/{v}.npz')]
        rate = sum(DUR[v] for v in got) / max(time.time() - t1, 1e-9)      # gồm cả nạp model: cận dưới của tốc độ
        print(f'5 video: {rate:.1f} video-s / s (gồm nạp model) -> còn khoảng {(total_sec - sum(DUR[v] for v in got)) / max(rate, 1e-9) / 3600:.2f} giờ')
        note(features={**INV['features'], 'video_sec_per_sec_5': rate, 'sec_per_video_min_5': (time.time() - t1) / max(len(got), 1)})
        extract(SRCV, FEATS, f'{B}/extract.log')
    FEAT_FAIL = [v for v in VIDS if not os.path.exists(f'{FEATS}/{v}.npz')]
    rows = {}
    for v in VIDS:
        if v not in FEAT_FAIL:
            with np.load(f'{FEATS}/{v}.npz') as z:
                rows[v] = [int(z['k768'].shape[0]), int(z['a768'].shape[0]), bool(np.abs(z['a768']).sum() > 0)]
    print(f'đặc trưng: {len(rows)} / {len(VIDS)} video | không trích được: {FEAT_FAIL} | a768 toàn 0: {[v for v, r in rows.items() if not r[2]]}')
    note(features={**INV['features'], 'videos': len(rows), 'failed': FEAT_FAIL, 'audio_all_zero': [v for v, r in rows.items() if not r[2]],
                   'rows_minus_duration_max': max(abs(r[0] - DUR[v]) for v, r in rows.items()) if rows else None, 'sec': round(time.time() - t0, 1)})
    check('mọi video của kho có đặc trưng', not FEAT_FAIL, str(FEAT_FAIL[:5]))
check('có đặc trưng cho mọi video của kho (RUN_FEATS = False thì phải đã có trên HF)', all(os.path.exists(f'{FEATS}/{v}.npz') for v in VIDS))
sync_up('ô 5: đặc trưng')

In [ ]:
# ===== 6. Plan §4, §5: UniAV (checkpoint và θ của configs/uemr_final.json), các dòng R0-R8, vector h_i =====
from uemr_htstep import final_bench as fb, final_dev as fd
CFG_P = f'{FUSHION}/configs/uemr_final.json'
check('sha256 configs/uemr_final.json (khoá ở file 02 §7)', sha256(CFG_P) == CONFIG_SHA, sha256(CFG_P)[:16])
CFG = json.load(open(CFG_P, encoding='utf-8'))
print('phương pháp chính:', CFG['main_method'], '| θ', CFG['params']['theta'], '| L_min', CFG['params']['l_min'], '| L_fill', CFG['params']['l_fill'], '| β*', CFG['params']['beta_star'])
t0 = time.time()
UNI, SEGD, STU = f'{B}/uniav', f'{B}/segments', f'{B}/student'
for d in (UNI, SEGD, STU, '/content/empty_omni/val'):
    os.makedirs(d, exist_ok=True)
CK_U = hf_hub_download(CFG['models']['uniav']['repo'], 'model_best.pth.tar', local_dir='/content/ckpt/uniav')
check('sha256 checkpoint UniAV = config', sha256(CK_U) == CFG['models']['uniav']['sha256'], sha256(CK_U)[:16])
HT = '/content/htstep_official'
if not os.path.isdir(f'{HT}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/facebookresearch/htstep', HT], check=True)
subprocess.run(['git', '-C', HT, 'checkout', '-q', D['annotations']['commit']], check=True)
TAX = f'{HT}/data/taxonomy.csv'
check('sha256 taxonomy.csv chính thức', sha256(TAX) == D['taxonomy']['sha256'])
STEP_EMB = hf_hub_download('nguyenminh04/htstep-text-feats', 'text_feats/step_emb_iv2j.npz', repo_type='dataset', local_dir='/content/ckpt/uniav')      # như lúc train (notebook 01 A3)
TAXR = sorted(csv.DictReader(open(TAX, newline='')), key=lambda r: int(r['global_step_index']))
ARTICLES = list(dict.fromkeys((r['activity'], r['variation']) for r in TAXR))      # thứ tự article của HTStepDataset

# video benchmark trong định dạng annotations.json của HT-Step để HTStepDataset (không sửa) đọc đặc trưng; bước giữ chỗ không tới head nào ở chế độ article đoán
def uniav_ann(path, row):
    wj(path, fb.uniav_annotations({v: float(BENCH[v]['duration']) for v in VIDS}, TAXR[row]['global_step_index']))
    return path
ANN_U = uniav_ann(f'{UNI}/annotations_bench.json', 0)
with open(f'{UNI}/videos.txt', 'w', encoding='utf-8') as f:
    f.write(''.join(f'{v}\n' for v in VIDS))
UA = f'--uniav-repo {HANLI} --taxonomy {TAX} --ckpt {CK_U} --step-emb {STEP_EMB} --feat-dir {FEATS} --omni-dir /content/empty_omni --videos {UNI}/videos.txt --device cuda'
def uniav(cmd, ann, out, extra=''):
    get_ipython().system(f'cd {FUSHION} && python -m uemr_htstep.run_uniav {cmd} {UA} --annotations "{ann}" --out "{out}" {extra} 2>&1 | tee -a "{UNI}/uniav.log"')
read_raw = lambda p: {json.loads(l)['video_id']: json.loads(l) for l in open(p, encoding='utf-8') if l.strip()}

uniav('proposals', ANN_U, UNI)
RAW = read_raw(f'{UNI}/uniav_raw.jsonl')
check('UniAV chạy đủ video của kho', set(VIDS) <= set(RAW), str(sorted(set(VIDS) - set(RAW))[:5]))
# CHECK: đổi bước giữ chỗ (bước khác, article khác) không đổi đề xuất và article đoán
shutil.rmtree(f'{A}/uniav_chk', ignore_errors=True)
uniav('proposals', uniav_ann(f'{A}/annotations_chk.json', -1), f'{A}/uniav_chk', '--limit 5')
CHK = read_raw(f'{A}/uniav_chk/uniav_raw.jsonl')
check('đổi bước giữ chỗ không đổi kết quả chế độ article đoán (5 video)',
      len(CHK) == min(5, len(VIDS)) and all(CHK[v]['pred'] == RAW[v]['pred'] and CHK[v]['article_top1'] == RAW[v]['article_top1'] for v in CHK))

ROWS, ROW_INFO = fb.build_rows({v: RAW[v]['pred'] for v in VIDS}, DUR, GT, CFG)
ROW_SHA = fd.write_rows(ROWS, SEGD)
ROW_INFO.update(files=ROW_SHA, uniav_raw_sha256=sha256(f'{UNI}/uniav_raw.jsonl'), videos_sha256=VIDS_SHA)
old = rj(f'{SEGD}/rows.json')
check('bộ đoạn trùng lượt trước (resume)', old is None or old['files'] == ROW_SHA, 'đoạn đổi so với lượt trước: xoá thư mục yc2 trên HF nếu cố ý làm lại')
wj(f'{SEGD}/rows.json', ROW_INFO)
QUAL = fb.quality(ROWS, GT, DUR, {v: RAW[v]['article_top1'] for v in VIDS}, [f'{a} / {b}' for a, b in ARTICLES])
wj(f'{RES}/uniav_quality.json', QUAL)
print('dòng:', json.dumps({k: v for k, v in ROW_INFO.items() if k not in ('files', 'videos_without_r6')}, ensure_ascii=False), '| video không có đoạn R6:', len(ROW_INFO['videos_without_r6']))
print('chất lượng cắt (chỉ mô tả):', json.dumps(QUAL, indent=1, ensure_ascii=False))

done = rj(f'{STU}/done.json', {})
if done.get('files') != ROW_SHA or not all(os.path.exists(f'{STU}/student_{n}.npz') for n in fb.NAMES):
    uniav('student', ANN_U, STU, f'--segments {SEGD} --names ' + ' '.join(fb.NAMES))
    check('đủ vector h_i của mọi dòng', all(os.path.exists(f'{STU}/student_{n}.npz') for n in fb.NAMES))
    wj(f'{STU}/done.json', {'files': ROW_SHA, 'sha256': {n: sha256(f'{STU}/student_{n}.npz') for n in fb.NAMES}})
note(uniav={'ckpt_sha256': CFG['models']['uniav']['sha256'], 'step_emb_sha256': sha256(STEP_EMB), 'rows': ROW_INFO, 'student_sha256': rj(f'{STU}/done.json')['sha256'],
            'sec': round(time.time() - t0, 1)})
sync_up('ô 6: UniAV, đoạn, h_i')

In [ ]:
# ===== 7. Plan §7: khoá protocol benchmark TRƯỚC khi chấm (ô 9 kiểm sha256) =====
PROTO = {'version': 1, 'benchmark': 'yc2', 'subset': YC2_SUBSET or 'training + validation', 'smoke': SMOKE, 'uemr_final_sha256': CONFIG_SHA, 'main_method': CFG['main_method'],
         'params': CFG['params'], 'annotation': INV['annotation'], 'videos': len(VIDS), 'videos_sha256': VIDS_SHA, 'queries': len(CAPS), 'queries_sha256': INV['queries_sha256'],
         'segments_sha256': ROW_SHA, 'chunk_sec_r2': ROW_INFO['chunk_sec'], 'clips': ROW_INFO['clips'], 'uniav_raw_sha256': ROW_INFO['uniav_raw_sha256'],
         'adapter_sha256': ADAPTER_SHA, 'multi_positive': CFG['rules']['multi_positive'], 'eval': CFG['eval'],
         'criteria': {'UEMR': 'phương pháp chính của uemr_final.json trên R8; multi-positive, R@1; đạt khi cận dưới CI 95 % > 0',
                      'K1': 'UEMR - R0 (b)', 'K2': 'UEMR - R3 cùng cột', 'K3': 'VCMR R6 (b) - VCMR R3 (b)', 'K4': 'UEMR-Lite R8 - R0 (b)', 'K5': '(U) - (b) trên R8',
                      'K6': 'chỉ báo: UEMR - R2, C1 - R6 (b), R7 - UEMR, pt - best (R0, R8)'},
         'rules': {'zscore': CFG['rules']['zscore'], 'r2_chunk': CFG['rules']['r2_chunk'], 'no_r6': 'video không có đề xuất nào >= θ: không có đoạn R6 / R3 (không truy hồi được ở hai dòng đó), R8 = đoạn lấp'},
         'fushion': COMMIT}
PP = f'{RES}/benchmark_protocol.json'
old = rj(PP)
if old is not None:
    diff = [k for k in PROTO if k not in ('fushion',) and old.get(k) != PROTO[k]]
    check('protocol đã khoá không đổi (resume)', not diff, f'khác ở {diff}')
else:
    wj(PP, {**PROTO, 'locked_at': time.strftime('%Y-%m-%d %H:%M:%S')})
PROTO_SHA = sha256(PP)
print('benchmark_protocol.json sha256', PROTO_SHA)
note(protocol_sha256=PROTO_SHA)
sync_up('ô 7: khoá protocol')

In [ ]:
# ===== 8. Plan §6: encode Omni như cột (b) của dev (mp4 + timestamps): best = mọi dòng + text; pt = R0, R8 + text (resume theo store) =====
from omni_retrieval.collect import collect_text
from omni_retrieval.manifest import segment_table, unique_segments, segment_records, caption_records, store_keys, write_jsonl
BASE_MODEL, BEATS = '/content/WAVE_HOME/WAVE-7B', '/content/WAVE_HOME/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt'
OM = lambda tag: f'{B}/omni/{tag}'
PARTS_OMNI = {'best': {n: ROWS[n] for n in fb.NAMES}, 'pt': {n: ROWS[n] for n in ('r0', 'r8')}}

def wave_ok():
    idx = glob.glob(f'{BASE_MODEL}/*.index.json')
    return os.path.exists(BEATS) and bool(idx) and all(os.path.exists(f'{BASE_MODEL}/{s}') for s in set(json.load(open(idx[0]))['weight_map'].values()))

def omni_env(tag):
    check('GPU đủ cho Omni 7B (>= 22 GB)', GPU_GB >= 22, f'{gpu}, {GPU_GB:.0f} GB')
    chk = subprocess.run([sys.executable, '-c', 'import decord, av, peft, torch, transformers; print(transformers.__version__, torch.cuda.is_available())'], capture_output=True, text=True)
    last = (chk.stdout.strip().splitlines() or [''])[-1]
    check('môi trường encode (transformers 4.51.3, CUDA)', chk.returncode == 0 and last.split()[:2] == ['4.51.3', 'True'], f'{last} {chk.stderr[-300:]}')
    if not wave_ok():
        sh('hf download nguyenminh04/omni-model --local-dir /content/WAVE_HOME --max-workers 2')
    check('WAVE-7B + BEATs đủ file', wave_ok())
    return f'cd {FUSHION} && python -m omni_retrieval.encode --base-model {BASE_MODEL} --beats-path {BEATS} --adapter {ADAPTERS[tag]} --video-root {VIDEOS}'

def encode(modality, manifest, store, log, enc, extra=''):
    with open(log, 'a') as f:
        f.write(f"\n===== {modality} | {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
    get_ipython().system(f'{enc} --modality {modality} --manifest "{manifest}" --store "{store}" {extra} 2>&1 | tee -a "{log}"')
    todo = [json.loads(l)['id'] for l in open(manifest) if l.strip()]
    return sorted(set(todo) - store_keys(store, modality))

caps_rec = [{'caption_id': c['caption_id'], 'sentence': c['sentence']} for c in CAPS]
OMNI_INFO = rj(f'{B}/omni/info.json', {})
if RUN_OMNI:
    for tag in OMNI_TAGS:
        t0 = time.time()
        AV, TX, DBD, log = f'{OM(tag)}/clips', f'{OM(tag)}/text', f'{OM(tag)}/db', f'{OM(tag)}/encode.log'
        os.makedirs(OM(tag), exist_ok=True)
        SEG_ROWS = segment_table(PARTS_OMNI[tag])
        UNIQ = unique_segments(SEG_ROWS)
        av_todo, tx_todo = segment_records(UNIQ, store_keys(AV, 'av')), caption_records(caps_rec, store_keys(TX, 'text'))
        print(f'[{tag}] clip cần encode: {len(av_todo)} / {len(UNIQ)} ({ {n: sum(len(s) for s in p.values()) for n, p in PARTS_OMNI[tag].items()} }) | câu hỏi: {len(tx_todo)} / {len(CAPS)} '
              f'| ước lượng {len(av_todo) * SEC_REF / 3600:.2f} giờ ở {SEC_REF} s/clip')
        info = {}
        if av_todo or tx_todo:
            enc = omni_env(tag)
            if av_todo:
                write_jsonl(f'{OM(tag)}/segs_todo.jsonl', av_todo)
                encode('av', f'{OM(tag)}/segs_todo.jsonl', AV, log, enc, '--num-workers 4')
                info['sec_per_clip'] = json.loads(open(f'{AV}/runs.jsonl').read().strip().splitlines()[-1]).get('sec_per_clip')
            if tx_todo:
                write_jsonl(f'{OM(tag)}/caps_todo.jsonl', tx_todo)
                left_t = encode('text', f'{OM(tag)}/caps_todo.jsonl', TX, log, enc, '--batch-size 32')
                check(f'{tag}: encode đủ câu hỏi', not left_t, str(left_t[:5]))
        left = [r['id'] for r in segment_records(UNIQ, store_keys(AV, 'av'))]
        err = {}
        if os.path.exists(f'{AV}/failed.jsonl'):
            for l in open(f'{AV}/failed.jsonl'):
                if l.strip():
                    q = json.loads(l); err[q['id']] = q['error']
        pd.DataFrame([{'seg_key': k, 'error': err.get(k, 'không có trong failed.jsonl')} for k in left], columns=['seg_key', 'error']).to_csv(f'{OM(tag)}/failed.csv', index=False)
        fail_pct = 100 * len(left) / len(UNIQ)
        check(f'{tag}: clip lỗi <= 2 %', fail_pct <= 2, f'{len(left)} clip ({fail_pct:.2f} %), xem {OM(tag)}/failed.csv')
        os.makedirs(DBD, exist_ok=True)      # vector đoạn đọc thẳng từ store lúc chấm; db chỉ giữ text.npz theo thứ tự câu hỏi
        txt = collect_text([{k: c[k] for k in ('caption_id', 'video_id', 'sentence', 'gt_ts', 'gt_te', 'gt_seg_key')} for c in CAPS], TX, f'{DBD}/text.npz')
        meta = {'tag': tag, 'partitions': sorted(PARTS_OMNI[tag]), 'segments_sha256': {n: ROW_SHA[n] for n in PARTS_OMNI[tag]}, 'videos_sha256': VIDS_SHA,
                'queries_sha256': INV['queries_sha256'], 'clips': len(UNIQ), 'text': txt, 'failed': left, **info, 'adapter_sha256': ADAPTER_SHA[tag], 'commit': COMMIT,
                'created': time.strftime('%Y-%m-%d %H:%M:%S'), 'vectors': 'fp16', 'text_sha256': sha256(f'{DBD}/text.npz'), 'sec': round(time.time() - t0, 1)}
        wj(f'{DBD}/meta.json', meta)
        OMNI_INFO[tag] = meta
        wj(f'{B}/omni/info.json', OMNI_INFO)
        print(f'[{tag}] clip {len(UNIQ)} (lỗi {len(left)}) | text {txt}')
        torch.cuda.empty_cache()
        sync_up(f'ô 8: Omni {tag}')

In [ ]:
# ===== 9. Plan §9: chấm MỘT LẦN (mọi dòng x cột x strict / multi, cặp + CI, tiêu chí K1-K6) =====
from omni_retrieval.evaluate import load_text
check('protocol đã khoá, sha256 không đổi', sha256(PP) == PROTO_SHA, PROTO_SHA[:16])
OUT_SC, SCORED_NOW = f'{RES}/score', False
if os.path.exists(f'{OUT_SC}/criteria.json'):
    print('ĐÃ CHẤM ở lượt trước: in lại, KHÔNG chấm lại.')
else:
    check('đã encode Omni đủ các tag', all(t in OMNI_INFO for t in OMNI_TAGS), str(sorted(OMNI_INFO)))
    t0 = time.time()
    PT = {'text': load_text(f'{OM("pt")}/db'), 'omni': fd.omni_vectors([], [f'{OM("pt")}/clips'])} if 'pt' in OMNI_TAGS else None
    fb.score(CAPS, load_text(f'{OM("best")}/db'), {n: rj(f'{SEGD}/{n}.json') for n in fb.NAMES}, fd.omni_vectors([], [f'{OM("best")}/clips']), STU, CFG, OUT_SC, PT)
    wj(f'{OUT_SC}/scored.json', {'at': time.strftime('%Y-%m-%d %H:%M:%S'), 'protocol_sha256': PROTO_SHA, 'fushion': COMMIT, 'smoke': SMOKE, 'sec': round(time.time() - t0, 1)})
    SCORED_NOW = True
    sync_up('ô 9: chấm')
MET, CRIT = rj(f'{OUT_SC}/metrics.json'), rj(f'{OUT_SC}/criteria.json')
T = pd.DataFrame(MET['table'])
for m in ('multi', 'strict'):
    print(f'== {m} | {MET["videos"]} video, {MET["queries"]} câu | UEMR = {MET["main_method"]}, β = {MET["beta"]}')
    display(T[T['mode'] == m].drop(columns=['mode', 'MnR']).round(2).reset_index(drop=True))
print(json.dumps({k: ({x: y for x, y in v.items() if x in ('cặp', 'A - B', 'CI', 'đạt')} if k[:2] in ('K1', 'K2', 'K3', 'K4', 'K5') else v) for k, v in CRIT.items()}, indent=1, ensure_ascii=False))

In [ ]:
# ===== 10. Tóm tắt cho Nhật ký file 03 =====
MAN = rj(f'{B}/run_manifest.json', {'runs': []})
MAN['runs'].append({'started': time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(T_START)), 'stage': 'pipeline + chấm', 'scored_this_run': SCORED_NOW, 'protocol_sha256': PROTO_SHA, 'flags': FLAGS, 'env': ENV, 'sec': round(time.time() - T_START, 1)})
wj(f'{B}/run_manifest.json', MAN)
print(json.dumps({k: INV.get(k) for k in ('annotation', 'filter_counts', 'overlap', 'videos_sha256', 'queries', 'queries_dropped', 'queries_sha256', 'queries_shared_text', 'limits')}, indent=1, ensure_ascii=False))
print('features:', json.dumps({k: v for k, v in (INV.get('features') or {}).items() if k != 'ckpt_sha256'}, indent=1, ensure_ascii=False))
print('uniav:', json.dumps({k: v for k, v in INV['uniav']['rows'].items() if k not in ('files', 'videos_without_r6')}, ensure_ascii=False), '| protocol', PROTO_SHA)
print('omni:', json.dumps({t: {k: (len(m[k]) if k == 'failed' else m.get(k)) for k in ('clips', 'text', 'failed', 'sec_per_clip', 'sec')} for t, m in OMNI_INFO.items()}, indent=1, ensure_ascii=False))
print('số lượt chấm trong run_manifest:', sum(bool(r.get('scored_this_run')) for r in MAN['runs']), '(phải là 1 khi SMOKE = False)')
sync_up('ô 10: tóm tắt')
if SMOKE:
    print('SMOKE xong: mọi CHECK OK. Đổi SMOKE = False ở ô 0 rồi Run all.')